# Suite DEF — Definitions

What a program declares, organized by meaning (`@mbse/programs/Framework/Definitions`): entities in scopes, with
lookup through transparent scopes, using-directives and bases. `Ccpp.Definitions.define` builds them from a C or C++
translation unit. Each case checks the entities, their declarations and definitions, and what names resolve to.

In [ ]:
import { readFileSync } from "node:fs";

import { Ccpp20, Definitions as D, Syntax as S } from "@mbse/programs/Ccpp";
import { CcppStandard } from "@mbse/programs/Ccpp/_Standard";
import { Definitions as FD, Syntax as F } from "@mbse/programs/Framework";

import { assert, equal } from "./support.js";

const { Entity, Program, Scope } = FD;
type Entity = FD.Entity;
const CPP26 = new CcppStandard(2026);

function define(text: string): [S.TranslationUnit, FD.Program] {
  const unit = CPP26.parse(text);
  return [unit, D.define(unit)];
}

const names = (unit: F.SyntaxNode, spelling: string) =>
  [...F.walk(unit)].filter((n): n is S.Identifier => n instanceof S.Identifier && n.spelling === spelling);
/** The same entities, in the same order. */
const identical = (a: readonly unknown[], b: readonly unknown[]) => a.length === b.length && a.every((x, i) => x === b[i]);
const one = (entities: Entity[]) => entities[0] as Entity;

## DEF-01 · Entities, scopes and lookup, on scopes built by hand

In [ ]:
const root = new Scope("namespace", null, null);
const outer = new Entity("namespace", "outer", root);
outer.scope = new Scope("namespace", outer, root);
root.declare(outer);
const inner = new Entity("class", "Inner", outer.scope);
inner.scope = new Scope("class", inner, outer.scope);
outer.scope.declare(inner);
const field = new Entity("field", "x", inner.scope);
inner.scope.declare(field);
assert(field.qualified_name() === "outer::Inner::x" && String(field) === "<field outer::Inner::x>");
assert(String(inner.scope) === "<class scope outer::Inner>" && String(root) === "<namespace scope>");
const unnamed = new Entity("namespace", null, root);
assert(String(unnamed) === "<namespace (unnamed)>" && unnamed.qualified_name() === "");
const dotted = new Scope("module", null, null, null, ".");
const klass = new Entity("class", "A", dotted);
klass.scope = new Scope("class", klass, dotted);
dotted.declare(klass);
const method = klass.scope.declare(new Entity("function", "b", klass.scope));
assert(klass.scope.separator === "." && method.qualified_name() === "A.b");
equal(new Program(dotted).lookup("A.b"), [method]);
assert(new Entity("variable", "free", null).qualified_name() === "free" && root.separator === "::");
const alias = new Entity("namespace alias", "o", root);
alias.target = outer;
root.declare(alias);
const again = new Entity("namespace alias", "p", root);
again.target = alias;
root.declare(again);
assert(again.resolved() === outer && outer.resolved() === outer);
const loop = new Entity("namespace alias", "q", root);
loop.target = loop;
assert(loop.resolved() === loop); // an alias of itself ends the chain
assert(identical(root.qualified(["o", "Inner", "x"]), [field]) && identical(root.qualified(["p", "Inner", "x"]), [field]));
assert(root.qualified([]).length === 0 && root.qualified(["x"]).length === 0 && root.qualified(["missing", "x"]).length === 0);
assert(identical(root.qualified(["outer", "Inner"], true), [inner]) && inner.scope.root() === root);
root.declare(field, "y");
root.declare(field, "y");
root.declare(new Entity("namespace", null, root));
assert(identical(root.names.get("y") as Entity[], [field]) && identical(root.entities(), [outer, alias, again, field]));
const base = new Scope("class", null, null);
const derived = new Scope("class", null, base);
base.declare(new Entity("field", "b", base));
derived.bases.push(base);
assert(identical(derived.lookup("b"), base.names.get("b") as Entity[]) && derived.lookup("c").length === 0);
const [first, second] = [new Scope("namespace", null, null), new Scope("namespace", null, null)];
first.transparent.push(second);
second.transparent.push(first);
second.declare(new Entity("variable", "v", second));
assert(identical(first.lookup("v"), second.names.get("v") as Entity[]) && first.lookup("w").length === 0); // cycles end
const used = new Scope("namespace", null, null);
used.declare(new Entity("variable", "u", used));
const block = new Scope("block", null, root);
block.using.push(used);
assert(identical(block.resolve("u"), used.names.get("u") as Entity[]) && identical(block.resolve("outer"), [outer]));
assert(block.resolve("none").length === 0);
let program = new Program(root);
assert(program.add(outer) === outer && identical([...program.entities()], [outer]));
const node = new S.Identifier({ spelling: "n" });
program.declares(node, outer);
program.located(node, block);
program.located(node, root); // the first scope recorded stays
assert(program.entity_of(node) === outer && program.scope_of(node) === block && program.entity_of(root) === null);
assert(identical(program.lookup("outer::Inner"), [inner]) && identical(program.lookup("::outer"), [outer]));
assert(identical(program.lookup("u", node), used.names.get("u") as Entity[]) && program.lookup("u").length === 0);
assert(program.scope_of(new S.Identifier({ spelling: "elsewhere" })) === null && program.macros.kind === "macros");
console.log("ok");

## DEF-02 · Namespaces: reopened, nested, inline, unnamed, aliased, used

In [ ]:
let unit: S.TranslationUnit;
[unit, program] = define("namespace a { int x; } namespace a { int y; } namespace a::b::inline c { int z; } "
  + "inline namespace v1 { int w; } namespace { int hidden; } namespace { int also; } "
  + "namespace n = a::b; namespace m = missing; namespace q = n; "
  + "namespace u { using namespace a; using namespace ::a; int k = x; } "
  + "using a::x; using none::y; enum class E { R, G }; void f() { using enum E; using enum E; R; } "
  + "namespace [[deprecated]] old {} using enum a;");
const a = one(program.lookup("a"));
assert(program.lookup("a").length === 1 && a.declarations.length === 3 && one((a.scope as FD.Scope).lookup("y")).kind === "variable");
assert(one(program.lookup("a::b::z")).qualified_name() === "a::b::c::z"); // through the inline namespace
assert(one(program.lookup("w")).qualified_name() === "v1::w" && one(program.lookup("hidden")).kind === "variable");
const unnamedNamespaces = [...program.entities()].filter((e) => e.kind === "namespace" && e.name === null);
assert(unnamedNamespaces.length === 1 && (unnamedNamespaces[0] as Entity).declarations.length === 2 && program.lookup("also").length > 0);
assert(program.entity_of(unit.items[4]) === unnamedNamespaces[0] && program.entity_of(unit.items[5]) === unnamedNamespaces[0]);
assert(one(program.lookup("n")).target === one(program.lookup("a::b")) && one(program.lookup("m")).target === null);
assert(one(program.lookup("q::z")).name === "z" && identical(program.lookup("n::z"), program.lookup("q::z")));
assert(identical((one(program.lookup("u")).scope as FD.Scope).using, [a.scope]));
assert(identical(D.referents(program, names(unit, "x").at(-2) as F.SyntaxNode), program.lookup("a::x"))); // through the directive
assert(identical(program.root.lookup("x"), program.lookup("a::x"))); // the using-declaration
const using = [...program.entities()].find((e) => e.kind === "using declaration") as Entity;
assert(using.name === "y" && using.parent === program.root);
equal(D.referents(program, names(unit, "R").at(-1) as F.SyntaxNode).map((e) => e.qualified_name()), ["E::R"]); // through `using enum`
console.log("ok");

## DEF-03 · Classes: members, bases, out-of-line definitions, unnamed classes and friends

In [ ]:
[unit, program] = define(`
struct B { int b; void g(); };
struct S;
struct S : B, missing::M {
    int field;
    static int count;
    static inline int ready = 1;
    static constexpr int size = 2;
    union { int i; float f; };
    struct { int depth; } info;
    void f() const;
    void f(int);
    friend class F;
    friend void g(S);
    friend void h() {}
};
int S::count = 0;
void S::f() const { field = b + size; }
void B::g() {}
struct A::Unknown { int u; };
typedef B Alias;
struct D : Alias { template <class T> friend void t(T); };
int S::extra = 3;
struct O { struct In; };
struct O::In { int z; };
`);
const s = one(program.lookup("S"));
assert(s.declarations.length === 2 && s.definition instanceof S.ClassSpecifier && s.definition.body !== null);
equal((s.scope as FD.Scope).entities().map((e) => [e.kind, e.name]), [["field", "field"], ["variable", "count"],
  ["variable", "ready"], ["variable", "size"], ["field", "info"], ["function", "f"], ["function", "f"], ["field", "extra"]]); // unnamed classes
const count = one(program.lookup("S::count"));
assert(count.declarations.length === 2 && count.definition === count.declarations[1]);
assert(one(program.lookup("S::ready")).definition !== null && one(program.lookup("S::size")).definition !== null);
assert(one(program.lookup("S::i")).kind === "field"); // an anonymous union's members are its class's
assert(program.lookup("S::depth").length === 0);
const constF = program.lookup("S::f").find((e) => e.signature === "() const") as Entity;
assert(constF.definition instanceof S.FunctionDefinition && constF.declarations.length === 2);
assert(one(program.lookup("B::g")).definition !== null);
assert(identical((s.scope as FD.Scope).bases, [one(program.lookup("B")).scope]));
assert(one(program.lookup("S::b")).parent?.owner?.name === "B");
assert(program.lookup("F").length === 0 && program.lookup("g").length === 0 && program.lookup("h").length === 0); // friends
assert(one((one(program.lookup("Unknown")).scope as FD.Scope).lookup("u")).kind === "field"); // declared where it is
assert((one(program.lookup("D")).scope as FD.Scope).bases.length === 0 && program.lookup("t").length === 0); // an alias
assert(one(program.lookup("S::extra")).parent === s.scope); // out of line, though the class does not declare it
assert(one(program.lookup("O::In::z")).kind === "field");
assert((one(program.lookup("O::In")).scope as FD.Scope).parent === one(program.lookup("O")).scope);
const sf = [...F.walk(unit)].filter((n) => n instanceof S.FunctionDefinition)[1] as F.SyntaxNode; // S::f
assert(identical(D.referents(program, names(sf, "field")[0] as F.SyntaxNode), program.lookup("S::field"))); // the class's scope
assert(identical(D.referents(program, names(sf, "b")[0] as F.SyntaxNode), program.lookup("B::b"))); // and its bases'
console.log("ok");

## DEF-04 · Enumerations, aliases, concepts, templates and macros

In [ ]:
[unit, program] = define(`
#define LIMIT 10
#define LIMIT 11
enum Color { Red, Green = LIMIT, };
enum class Mode : int { Off,
#ifdef X
  On,
#endif
};
enum Fwd : int;
typedef int Int, *IntPointer;
using Real = double;
template <typename T> concept Small = sizeof(T) < 4;
template <class T, int N, template <class> class W, typename... Ts>
struct Box { T items[N]; };
template <class, int> struct Unnamed;
template <class T> struct Box<T *> {};
template <class T> T identity(T t) { return t; }
template <class T> using Ptr = T *;
enum { Loose };
`);
equal(program.macros.entities().map((e) => e.kind), ["macro"]);
assert(one(program.macros.names.get("LIMIT") as Entity[]).declarations.length === 2);
assert(one(program.lookup("Red")).qualified_name() === "Color::Red" && program.lookup("Off").length === 0);
assert(one(program.lookup("Mode::On")).kind === "enumerator" && one(program.lookup("Fwd")).definition === null);
equal([...program.lookup("Int"), ...program.lookup("IntPointer"), ...program.lookup("Real")].map((e) => e.kind),
  ["type alias", "type alias", "type alias"]);
assert(one(program.lookup("Small")).kind === "concept");
const box = one(program.lookup("Box"));
assert(box.declarations.length === 2 && (box.definition as any).body.items.length > 0); // a specialization is a declaration
const template = unit.items[8] as F.SyntaxNode;
assert(program.entity_of(template) === box);
const parameters = (box.scope as FD.Scope).parent as FD.Scope;
assert(parameters.kind === "template");
equal(parameters.entities().map((e) => e.name), ["T", "N", "W", "Ts"]);
assert(one(D.referents(program, names(template, "T").at(-1) as F.SyntaxNode)).kind === "template parameter");
assert(program.entity_of(unit.items[11]) === one(program.lookup("identity")));
assert(program.entity_of(unit.items[12]) === one(program.lookup("Ptr")));
assert(program.scope_of((unit.items[9] as S.TemplateDeclaration).parameters[0])?.kind === "template");
assert(program.entity_of(unit.items[7]) === one(program.lookup("Small")));
assert(one(program.lookup("Loose")).parent?.owner?.name === null); // an unnamed enumeration's
console.log("ok");

## DEF-05 · Functions: overloads, parameters, blocks, statement scopes, labels, lambdas and handlers

In [ ]:
[unit, program] = define(`
void f(int);
void f(int x) { }
void f(double, ...);
int (*pointer)(int);
extern int declared;
int declared = 1;
auto [first, second] = pair;
void body(int a, int) {
    int a2 = a;
    { int inner = a2; }
    if (int c = a; c) { int d = c; }
    for (int i = 0; i < a; ++i) {}
    for (auto [k, v] : map) {}
start:
    try { } catch (const E &e) { use(e); } catch (...) {}
    auto l = [a, n = 1]<typename T, int K, class>(T t, int) { return t + n + a; };
    auto r = requires (int q) { q; };
    goto start;
}
`);
equal(program.lookup("f").map((e) => [e.signature, e.declarations.length, e.definition !== null]),
  [["(int)", 2, true], ["(double, ...)", 1, false]]);
assert(one(program.lookup("pointer")).kind === "variable"); // a pointer to a function is an object
const declared = one(program.lookup("declared"));
assert(declared.declarations.length === 2 && declared.definition === declared.declarations[1]);
equal([...program.lookup("first"), ...program.lookup("second")].map((e) => e.kind), ["variable", "variable"]);
const functionScope = program.scope_of((unit.items.at(-1) as S.FunctionDefinition).body) as FD.Scope;
assert(functionScope.kind === "function" && functionScope.owner === one(program.lookup("body")));
equal(functionScope.entities().map((e) => [e.kind, e.name]), [["parameter", "a"], ["label", "start"]]);
const kinds: Record<string, [string, string]> = {};
for (const name of ["a2", "inner", "c", "d", "i", "k", "v", "e", "n", "T", "K", "t", "q"]) {
  const found = [...program.entities()].find((x) => x.name === name) as Entity;
  kinds[name] = [found.kind, (found.parent as FD.Scope).kind];
}
equal(kinds, { a2: ["variable", "block"], inner: ["variable", "block"], c: ["variable", "block"], d: ["variable", "block"],
  i: ["variable", "block"], k: ["variable", "block"], v: ["variable", "block"], e: ["variable", "block"],
  n: ["variable", "lambda"], T: ["template parameter", "lambda"], K: ["template parameter", "lambda"],
  t: ["parameter", "lambda"], q: ["parameter", "requires"] });
const lambda = [...F.walk(unit)].find((x) => x instanceof S.LambdaExpression) as S.LambdaExpression;
assert(identical(D.referents(program, names(lambda.body as F.SyntaxNode, "a")[0] as F.SyntaxNode),
  [...program.entities()].filter((x) => x.kind === "parameter" && x.name === "a")));
assert(D.referents(program, new S.Identifier({ spelling: "x" })).length === 0); // a name outside the program
const handmade = new S.TranslationUnit({ items: [
  new S.SimpleDeclaration({ specifiers: [new S.PrimitiveTypeSpecifier({ keyword: "int" })],
    declarators: [new S.InitDeclarator({ bitfield: new S.IntegerLiteral({ spelling: "3" }) })] }),
  new S.FunctionDefinition({ declarator: new S.IdDeclarator({ name: new S.Identifier({ spelling: "g" }) }), body: new S.CompoundStatement() }),
  new S.SimpleDeclaration({ specifiers: [new S.PrimitiveTypeSpecifier({ keyword: "int" })], declarators: [new S.InitDeclarator({
    declarator: new S.IdDeclarator({ name: new S.QualifiedName({ qualifiers: [new S.DecltypeSpecifier({ expression: new S.ThisExpression() })],
      name: new S.Identifier({ spelling: "m" }) }) }) })] })] });
equal([...D.define(handmade).entities()].map((x) => [x.kind, x.qualified_name(), x.signature]),
  [["function", "g", null], ["variable", "m", null]]);
// a declarator that declares no name declares nothing, a definition without a function declarator has no signature,
// and a name qualified by a decltype is declared where it is
console.log("ok");

## DEF-06 · Names resolve from where they are, but members after `.` and `->` do not

In [ ]:
[unit, program] = define(`
int x;
namespace ns { int y; }
struct P { int x; template <class T> T get(); ~P(); };
void f(P *p, P q) {
    int z = p->x + x + q.template get<P>() + ns::y + ::x + p->P::x + decltype(q)::x;
    p->~P();
}
`);
const [member, plain] = names(unit, "x").slice(2, 4) as [F.SyntaxNode, F.SyntaxNode]; // `p->x`, then `x`
assert(program.scope_of(member) === null && D.referents(program, member).length === 0);
assert(identical(D.referents(program, plain), program.lookup("x")) && one(program.lookup("x")).parent === program.root);
const qualified = [...F.walk(unit)].filter((x) => x instanceof S.QualifiedName);
equal(qualified.map((q) => D.referents(program, q).map((x) => x.qualified_name())), [["ns::y"], ["x"], [], []]);
assert(program.scope_of(qualified[2] as F.SyntaxNode) === null); // `p->P::x`: a member
const templateId = [...F.walk(unit)].find((x) => x instanceof S.TemplateId) as S.TemplateId;
assert(program.scope_of(templateId) === null && program.scope_of(templateId.arguments[0] as F.SyntaxNode) !== null);
const destructor = [...F.walk(unit)].filter((x) => x instanceof S.DestructorName).at(-1) as S.DestructorName;
assert(identical(D.referents(program, destructor.type as F.SyntaxNode), program.lookup("P")));
console.log("ok");

## DEF-07 · Every entity of the conformance source, and every syntax node located

In [ ]:
unit = Ccpp20.parse(readFileSync("../../conformance/sources/cpp20.cpp", "utf8"));
program = D.define(unit);
assert([...program.entities()].length > 100);
assert(one(program.lookup("geo::v1::Point::x")).kind === "field");
assert(identical(program.lookup("geo::Point"), program.lookup("geo::v1::Point"))); // v1 is inline
equal([...new Set([...F.walk(unit)].filter((x) => program.scope_of(x) === null).map((x) => x.kind().KIND))].sort(),
  ["DestructorName", "Identifier", "TemplateId"]); // members after . and ->
assert(D.name_of(new S.DestructorName({ type: new S.DecltypeSpecifier({ expression: new S.ThisExpression() }) })) === "~decltype(this)");
assert(D.name_of(new S.ConversionName({ type: new S.TypeId({ specifiers: [new S.PrimitiveTypeSpecifier({ keyword: "int" })] }) }))
  === "operator int");
console.log("ok");